# 🏥 Predição de Custos de Seguro Médico
## MVP — Sistema de Suporte à Tomada de Decisão em Engenharia de Produção

---

| Campo | Informação |
|---|---|
| **Aluna** | Maria Gabriela Soares Canabrava Mascarenhas |
| **Matrícula** | 202043405 |
| **Universidade** | Universidade de Brasília (UnB) |
| **Departamento** | Engenharia de Produção |
| **Disciplina** | Sistema de Suporte à Tomada de Decisão |
| **Professor** | Dr. André Luiz Marques Serrano |
| **Data** | Abril de 2026 |

---

## Resumo

Este notebook apresenta o desenvolvimento completo de um modelo preditivo de **regressão supervisionada** para estimar o custo de planos de seguro médico com base em características pessoais e comportamentais dos segurados.

O trabalho segue o pipeline completo de Machine Learning: da definição do problema e análise exploratória, passando pela preparação dos dados com pipelines sklearn, treinamento e comparação de múltiplos modelos (Regressão Linear, Ridge, Random Forest e Gradient Boosting), otimização de hiperparâmetros com GridSearchCV, até a avaliação final com métricas de regressão e análise de feature importance — conectando os resultados ao contexto de Suporte à Decisão na indústria de seguros.

---


---
## 0. Configuração do Ambiente

Antes de iniciar, instalamos e importamos todas as bibliotecas necessárias para execução do notebook.


In [ ]:
# Instalação de dependências extras (caso necessário no Colab)
# !pip install -q scikit-learn pandas numpy matplotlib seaborn scipy


In [ ]:
# ─── Importações ────────────────────────────────────────────────────────────

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Pré-processamento e pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Seleção de features
from sklearn.feature_selection import SelectKBest, f_regression

# Modelos
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Validação e métricas
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Testes estatísticos
from scipy import stats

# Reprodutibilidade e formatação
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Configuração visual global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print("✅ Ambiente configurado com sucesso!")


---
## 1. Definição do Problema

### 1.1 Contexto e Objetivo

O setor de seguros de saúde enfrenta um desafio central de **precificação de risco**: definir o valor justo do prêmio cobrado de cada segurado sem comprometer a sustentabilidade financeira da operadora. Cobrar pouco aumenta o risco de sinistralidade; cobrar demais reduz a competitividade.

O objetivo deste trabalho é desenvolver um **modelo preditivo de regressão** capaz de estimar o custo individual do seguro médico (`charges`) com base em atributos demográficos e comportamentais do segurado.

**Pergunta central:**
> *Com base no perfil de um cliente (idade, IMC, hábito de fumar, número de dependentes e região), é possível estimar com boa acurácia o custo do seu seguro médico?*

---

### 1.2 Tipo de Tarefa

- **Tarefa:** Regressão supervisionada
- **Variável-alvo:** `charges` — valor cobrado pelo plano de seguro médico (US$), contínua e positiva
- **Modelos avaliados:** Regressão Linear, Ridge Regression, Random Forest Regressor, Gradient Boosting Regressor

---

### 1.3 Hipóteses Iniciais

Antes da análise, formulamos as seguintes hipóteses com base em conhecimento de domínio:

| # | Hipótese | Justificativa |
|---|---|---|
| H1 | Fumantes pagam significativamente mais | Maior risco de doenças crônicas e oncológicas |
| H2 | Idade tem relação positiva com o custo | Risco de saúde acumulado ao longo da vida |
| H3 | IMC elevado eleva o custo | Associado a doenças crônicas (diabetes, hipertensão) |
| H4 | Mais dependentes elevam o custo moderadamente | Maior cobertura contratada |
| H5 | Região e sexo têm influência menor | Fatores de risco clínicos devem predominar |

Essas hipóteses serão testadas via análise exploratória e pela importância de atributos nos modelos treinados.

---

### 1.4 Descrição do Dataset

- **Nome:** Medical Cost Personal Dataset
- **Fonte:** Repositório PyCaret / Kaggle
- **Registros:** 1.338 linhas
- **Atributos:** 7 colunas (6 preditores + 1 variável-alvo)
- **Valores faltantes:** Nenhum

| Atributo | Tipo | Descrição |
|---|---|---|
| `age` | Numérico | Idade do segurado (anos) |
| `sex` | Categórico | Sexo (male / female) |
| `bmi` | Numérico | Índice de massa corporal |
| `children` | Numérico | Nº de filhos dependentes |
| `smoker` | Categórico | Fumante (yes / no) |
| `region` | Categórico | Região dos EUA (4 regiões) |
| `charges` | **Numérico (alvo)** | **Custo do seguro médico (US$)** |

---

### 1.5 Restrições de Seleção

- Dataset não utilizado nas disciplinas da sprint
- Carregado diretamente via URL (sem configuração manual)
- Totalmente público, licença livre para uso educacional
- Sem valores faltantes, adequado para demonstrar todo o pipeline de ML


---
## 2. Carregamento e Inspeção Inicial dos Dados

O dataset é carregado diretamente via URL pública, garantindo que o notebook possa ser executado em qualquer ambiente sem configuração adicional.


In [ ]:
# ─── Carregamento do dataset via URL ────────────────────────────────────────
URL = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/insurance.csv"
df = pd.read_csv(URL)

print(f"✅ Dataset carregado com sucesso!")
print(f"   Shape: {df.shape[0]} linhas × {df.shape[1]} colunas")


In [ ]:
# ─── Primeiras linhas ────────────────────────────────────────────────────────
df.head(10)


In [ ]:
# ─── Tipos de dados e valores faltantes ─────────────────────────────────────
print("📋 Informações do Dataset:")
print(f"{'Coluna':<12} {'Tipo':<15} {'Nulos':<8} {'Únicos'}")
print("-" * 50)
for col in df.columns:
    print(f"{col:<12} {str(df[col].dtype):<15} {df[col].isnull().sum():<8} {df[col].nunique()}")


In [ ]:
# ─── Estatísticas descritivas ────────────────────────────────────────────────
print("📊 Estatísticas Descritivas — Variáveis Numéricas:")
df.describe().round(2)


In [ ]:
# ─── Distribuição das variáveis categóricas ──────────────────────────────────
print("📊 Distribuição das Variáveis Categóricas:\n")
for col in ['sex', 'smoker', 'region']:
    print(f"  {col.upper()}:")
    print(df[col].value_counts().to_string())
    print()


---
## 3. Análise Exploratória de Dados (EDA)

A EDA tem o objetivo de entender a estrutura dos dados, identificar padrões, outliers e relações entre as variáveis — especialmente em relação à variável-alvo `charges`.

### 3.1 Distribuição da Variável-Alvo (`charges`)

Um aspecto crítico em problemas de regressão de custos é a distribuição da variável-alvo. Custos financeiros frequentemente seguem uma **distribuição log-normal** — assimétrica à direita, com cauda longa.

Aplicaremos a transformação **log₁(1 + x)** (`log1p`) para aproximar a distribuição de uma normal, o que melhora o desempenho de modelos lineares e reduz o impacto de outliers.


In [ ]:
# ─── Distribuição de charges: original vs log-transformada ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histograma original
axes[0].hist(df['charges'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('charges — Distribuição Original', fontweight='bold')
axes[0].set_xlabel('Custo (US$)')
axes[0].set_ylabel('Frequência')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Histograma log-transformado
log_charges = np.log1p(df['charges'])
axes[1].hist(log_charges, bins=50, color='seagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('log(charges) — Após Transformação', fontweight='bold')
axes[1].set_xlabel('log(1 + Custo)')
axes[1].set_ylabel('Frequência')

# Q-Q Plot para verificar normalidade após log-transform
stats.probplot(log_charges, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot — log(charges)', fontweight='bold')

plt.tight_layout()
plt.suptitle('Figura 1 — Análise da Variável-Alvo: charges', y=1.02, fontsize=13, fontweight='bold')
plt.show()

# Teste de normalidade (Shapiro-Wilk na amostra)
sample = log_charges.sample(200, random_state=42)
stat, p = stats.shapiro(sample)
print(f"\n📐 Teste de Shapiro-Wilk em log(charges) (n=200):")
print(f"   Estatística W = {stat:.4f} | p-valor = {p:.4f}")
print(f"   → {'Não rejeitamos' if p > 0.05 else 'Rejeitamos'} H₀ de normalidade (α=0.05)")


### 3.2 Relação entre Preditores e a Variável-Alvo

Analisamos como cada variável preditora se relaciona com `charges`, verificando as hipóteses H1–H5.


In [ ]:
# ─── Relação de variáveis numéricas com charges ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
num_cols = ['age', 'bmi', 'children']
titles   = ['Idade (age)', 'IMC (bmi)', 'Nº de Dependentes (children)']

for ax, col, title in zip(axes, num_cols, titles):
    ax.scatter(df[col], df['charges'], alpha=0.3, s=18, color='steelblue')
    ax.set_xlabel(title)
    ax.set_ylabel('charges (US$)')
    ax.set_title(f'{title} × charges', fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Figura 2 — Variáveis Numéricas vs. charges', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─── Impacto de smoker, sex e region em charges ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cat_cols = ['smoker', 'sex', 'region']

for ax, col in zip(axes, cat_cols):
    order = df.groupby(col)['charges'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='charges', order=order, ax=ax, palette='Set2')
    ax.set_title(f'{col.upper()} vs. charges', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('charges (US$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Figura 3 — Variáveis Categóricas vs. charges', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Diferença de média entre fumantes e não fumantes
mean_smoker    = df[df['smoker'] == 'yes']['charges'].mean()
mean_nonsmoker = df[df['smoker'] == 'no']['charges'].mean()
print(f"\n💡 Média de charges:")
print(f"   Fumantes:     US$ {mean_smoker:,.2f}")
print(f"   Não fumantes: US$ {mean_nonsmoker:,.2f}")
print(f"   Razão: {mean_smoker/mean_nonsmoker:.1f}x maior para fumantes → confirma H1")


In [ ]:
# ─── Matriz de correlação (variáveis numéricas) ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

corr = df[['age', 'bmi', 'children', 'charges']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Figura 4 — Matriz de Correlação (Variáveis Numéricas)', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Correlações com charges:")
print(corr['charges'].drop('charges').sort_values(ascending=False).round(3).to_string())


In [ ]:
# ─── Interação smoker × age × charges ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'yes': '#e74c3c', 'no': '#3498db'}
for smoker_val, group in df.groupby('smoker'):
    ax.scatter(group['age'], group['charges'],
               c=colors[smoker_val], alpha=0.4, s=18,
               label=f"Fumante: {smoker_val}")

ax.set_xlabel('Idade (age)')
ax.set_ylabel('charges (US$)')
ax.set_title('Figura 5 — Interação: Idade × charges (por status de fumante)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()


### 3.3 Síntese da EDA — Verificação das Hipóteses

| Hipótese | Resultado | Evidência |
|---|---|---|
| H1 — Fumantes pagam mais | ✅ **Confirmada** | Média ~3,8× maior; maior separação nos boxplots |
| H2 — Idade eleva o custo | ✅ **Confirmada** | Correlação positiva de 0.299; três faixas visíveis no scatter |
| H3 — IMC elevado eleva o custo | ✅ **Confirmada (parcialmente)** | Correlação 0.198; efeito amplificado em fumantes |
| H4 — Dependentes elevam custo moderadamente | ✅ **Confirmada** | Correlação baixa (0.068), efeito pequeno mas positivo |
| H5 — Região e sexo têm influência menor | ✅ **Confirmada** | Boxplots sem diferença expressiva entre grupos |

**Decisão:** aplicaremos `log1p(charges)` como variável-alvo nos modelos para estabilizar a variância e melhorar o ajuste de modelos lineares.


---
## 4. Preparação dos Dados

### 4.1 Separação Treino / Teste

Separamos 80% dos dados para treino e 20% para teste final, com `random_state=42` para garantir reprodutibilidade. A base de teste permanece **completamente isolada** durante toda a fase de modelagem — somente utilizada na avaliação final, evitando data leakage.


In [ ]:
# ─── Definição de X e y ──────────────────────────────────────────────────────
X = df.drop('charges', axis=1)
y = np.log1p(df['charges'])   # log-transform da variável-alvo

# ─── Split treino/teste ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Separação concluída:")
print(f"   Treino:  {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Teste:   {X_test.shape[0]}  amostras ({X_test.shape[0]/len(X)*100:.0f}%)")


### 4.2 Definição das Transformações por Tipo de Variável

Aplicamos transformações distintas conforme o tipo de cada variável:

- **Variáveis numéricas** (`age`, `bmi`, `children`): padronização com `StandardScaler` (média 0, desvio padrão 1), necessário para modelos sensíveis à escala como a Regressão Linear e Ridge.
- **Variáveis categóricas** (`sex`, `smoker`, `region`): codificação com `OneHotEncoder` com `drop='first'` para evitar multicolinearidade (dummy variable trap).

O pré-processador é encapsulado em um `ColumnTransformer`, que será integrado aos pipelines dos modelos — garantindo que o `fit` ocorra **apenas no conjunto de treino**, sem vazamento de informação para o teste.


In [ ]:
# ─── Definição das colunas por tipo ─────────────────────────────────────────
numeric_features     = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

# ─── Pré-processador via ColumnTransformer ────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
])

print("✅ Pré-processador definido:")
print(f"   Numéricas    ({len(numeric_features)}): {numeric_features}")
print(f"   Categóricas  ({len(categorical_features)}): {categorical_features}")


### 4.3 Validação Cruzada

Utilizamos **K-Fold cross-validation com k=5** sobre o conjunto de treino. Isso divide o treino em 5 partes iguais, treina em 4 e valida em 1 por rodada, repetindo 5 vezes. As vantagens são:

- Estimativa mais confiável do desempenho real (reduz variância do estimador)
- Melhor uso dos dados — todos os exemplos participam do treino e da validação
- Permite reportar **média ± desvio-padrão** do R², demonstrando rigor estatístico

O uso de cross-validation é justificado pelo tamanho moderado do dataset (1.338 registros), onde uma única divisão treino/validação teria variância elevada.


In [ ]:
# ─── Configuração do K-Fold ──────────────────────────────────────────────────
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
print("✅ K-Fold configurado: 5 folds, shuffle=True, random_state=42")


---
## 5. Modelagem e Treinamento

### 5.1 Justificativa dos Algoritmos Selecionados

Selecionamos quatro algoritmos com grau crescente de complexidade:

| Algoritmo | Justificativa |
|---|---|
| **Regressão Linear** | Baseline interpretável; supõe relação linear entre preditores e alvo |
| **Ridge Regression** | Regressão linear com regularização L2 — reduz overfitting em preditores correlacionados |
| **Random Forest** | Ensemble de árvores de decisão; captura relações não-lineares e interações entre variáveis |
| **Gradient Boosting** | Ensemble sequencial que minimiza o erro residual iterativamente; geralmente o mais preciso |

A estratégia é partir de um **baseline simples** (Regressão Linear) e avançar em complexidade, comparando os ganhos de desempenho.

### 5.2 Treinamento com Pipeline + Cross-Validation


In [ ]:
# ─── Definição dos modelos ───────────────────────────────────────────────────
models = {
    'Regressão Linear':   LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, random_state=42)
}

# ─── Treinamento e avaliação via cross-validation ────────────────────────────
cv_results = {}
fitted_pipes = {}

print("🔄 Treinando modelos com cross-validation (k=5)...\n")

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Cross-validation no treino
    cv_r2   = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='r2')
    cv_rmse = cross_val_score(pipe, X_train, y_train, cv=kfold,
                              scoring='neg_root_mean_squared_error')

    # Treino final no conjunto de treino completo
    pipe.fit(X_train, y_train)
    fitted_pipes[name] = pipe

    cv_results[name] = {
        'CV R² (média)':  cv_r2.mean(),
        'CV R² (±std)':   cv_r2.std(),
        'CV RMSE (média)': -cv_rmse.mean(),
        'CV RMSE (±std)':  cv_rmse.std()
    }

    print(f"  ✅ {name:<22} | CV R² = {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

print("\n✅ Todos os modelos treinados!")


In [ ]:
# ─── Tabela de resultados do cross-validation ────────────────────────────────
cv_df = pd.DataFrame(cv_results).T.round(4)
print("📊 Resultados — Cross-Validation (k=5) no Conjunto de Treino:")
cv_df


---
## 6. Otimização de Hiperparâmetros

Com base nos resultados da cross-validation, identificamos os modelos mais promissores e aplicamos **GridSearchCV** para otimizar seus hiperparâmetros. O GridSearchCV realiza busca exaustiva sobre um grid de combinações, avaliando cada combinação via cross-validation interna — sem tocar no conjunto de teste.

### 6.1 Otimização do Random Forest


In [ ]:
# ─── Grid Search — Random Forest ─────────────────────────────────────────────
param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth':    [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

grid_rf = GridSearchCV(
    pipe_rf, param_grid_rf,
    cv=kfold, scoring='r2',
    n_jobs=-1, verbose=0
)

print("🔄 Otimizando Random Forest (GridSearchCV)...")
grid_rf.fit(X_train, y_train)

print(f"✅ Melhores hiperparâmetros — Random Forest:")
for param, val in grid_rf.best_params_.items():
    print(f"   {param}: {val}")
print(f"   Melhor CV R²: {grid_rf.best_score_:.4f}")


### 6.2 Otimização do Gradient Boosting


In [ ]:
# ─── Grid Search — Gradient Boosting ─────────────────────────────────────────
param_grid_gb = {
    'model__n_estimators':  [100, 200, 300],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_depth':     [3, 5]
}

pipe_gb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

grid_gb = GridSearchCV(
    pipe_gb, param_grid_gb,
    cv=kfold, scoring='r2',
    n_jobs=-1, verbose=0
)

print("🔄 Otimizando Gradient Boosting (GridSearchCV)...")
grid_gb.fit(X_train, y_train)

print(f"✅ Melhores hiperparâmetros — Gradient Boosting:")
for param, val in grid_gb.best_params_.items():
    print(f"   {param}: {val}")
print(f"   Melhor CV R²: {grid_gb.best_score_:.4f}")


---
## 7. Avaliação Final nos Dados de Teste

### 7.1 Métricas Utilizadas e Justificativas

Para avaliar modelos de regressão de custos, utilizamos três métricas complementares:

| Métrica | Fórmula | Interpretação |
|---|---|---|
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporção da variância de `charges` explicada pelo modelo. Quanto mais próximo de 1, melhor. |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Erro médio em escala da variável-alvo. Penaliza erros grandes. |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Erro absoluto médio. Mais robusto a outliers que o RMSE. |

> **Nota:** como aplicamos `log1p` na variável-alvo, as métricas são calculadas no espaço logarítmico. Para interpretação em dólares, aplicamos `expm1` nas predições.

### 7.2 Resultados no Conjunto de Teste


In [ ]:
# ─── Avaliação de todos os modelos no conjunto de teste ──────────────────────

# Incluímos os modelos otimizados junto aos originais
all_models = {
    **fitted_pipes,
    'Random Forest (Otimizado)':     grid_rf.best_estimator_,
    'Gradient Boosting (Otimizado)': grid_gb.best_estimator_
}

test_results = {}

for name, pipe in all_models.items():
    y_pred_log = pipe.predict(X_test)

    # Métricas no espaço log
    r2   = r2_score(y_test, y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))
    mae  = mean_absolute_error(y_test, y_pred_log)

    # RMSE em dólares (espaço original)
    y_pred_orig = np.expm1(y_pred_log)
    y_test_orig = np.expm1(y_test)
    rmse_usd = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
    mae_usd  = mean_absolute_error(y_test_orig, y_pred_orig)

    test_results[name] = {
        'R² (teste)': round(r2, 4),
        'RMSE (log)': round(rmse, 4),
        'MAE (log)':  round(mae, 4),
        'RMSE (US$)': round(rmse_usd, 2),
        'MAE (US$)':  round(mae_usd, 2)
    }

results_df = pd.DataFrame(test_results).T.sort_values('R² (teste)', ascending=False)
print("📊 Avaliação Final — Conjunto de Teste:")
results_df


In [ ]:
# ─── Gráfico comparativo de R² entre modelos ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6', '#1abc9c']
r2_vals = results_df['R² (teste)'].values
names   = results_df.index.tolist()

bars = ax.barh(names, r2_vals, color=colors[:len(names)], edgecolor='white', height=0.55)

for bar, val in zip(bars, r2_vals):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('R² (Conjunto de Teste)')
ax.set_title('Figura 6 — Comparação de R² entre Modelos (Conjunto de Teste)', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.axvline(x=0.9, color='gray', linestyle='--', alpha=0.5, label='R²=0.90')
ax.legend()
plt.tight_layout()
plt.show()


### 7.3 Análise de Overfitting

Verificamos se há overfitting comparando o desempenho na cross-validation (treino) com o desempenho no teste.


In [ ]:
# ─── Verificação de overfitting (CV vs Teste) ────────────────────────────────
print(f"{'Modelo':<32} {'CV R² (médio)':<18} {'Teste R²':<14} {'Diferença'}")
print("-" * 75)

modelos_base = ['Regressão Linear', 'Ridge Regression', 'Random Forest', 'Gradient Boosting']
for name in modelos_base:
    cv_r2   = cv_results[name]['CV R² (média)']
    test_r2 = test_results[name]['R² (teste)']
    diff    = test_r2 - cv_r2
    flag    = "⚠️" if abs(diff) > 0.05 else "✅"
    print(f"{flag} {name:<30} {cv_r2:<18.4f} {test_r2:<14.4f} {diff:+.4f}")


### 7.4 Análise dos Resíduos do Melhor Modelo

Analisamos os resíduos do melhor modelo para verificar premissas de uma boa regressão: resíduos próximos de zero, sem padrão sistemático e aproximadamente normais.


In [ ]:
# ─── Identificar melhor modelo e analisar resíduos ───────────────────────────
best_name = results_df.index[0]
best_pipe = all_models[best_name]

y_pred_log = best_pipe.predict(X_test)
residuals  = y_test.values - y_pred_log

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predito vs Real
axes[0].scatter(y_test, y_pred_log, alpha=0.4, s=18, color='steelblue')
lims = [min(y_test.min(), y_pred_log.min()), max(y_test.max(), y_pred_log.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Predição perfeita')
axes[0].set_xlabel('log(charges) — Real')
axes[0].set_ylabel('log(charges) — Predito')
axes[0].set_title('Predito vs Real', fontweight='bold')
axes[0].legend()

# 2. Resíduos vs Predito
axes[1].scatter(y_pred_log, residuals, alpha=0.4, s=18, color='seagreen')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('log(charges) — Predito')
axes[1].set_ylabel('Resíduo')
axes[1].set_title('Resíduos vs Predito', fontweight='bold')

# 3. Distribuição dos resíduos
axes[2].hist(residuals, bins=40, color='salmon', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='darkred', linestyle='--', lw=1.5)
axes[2].set_xlabel('Resíduo')
axes[2].set_ylabel('Frequência')
axes[2].set_title('Distribuição dos Resíduos', fontweight='bold')

plt.suptitle(f'Figura 7 — Análise de Resíduos: {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📐 Estatísticas dos Resíduos ({best_name}):")
print(f"   Média:         {residuals.mean():.6f}  (ideal: ≈ 0)")
print(f"   Desvio-padrão: {residuals.std():.4f}")
print(f"   Mín / Máx:     {residuals.min():.4f}  /  {residuals.max():.4f}")


---
## 8. Interpretabilidade — Feature Importance

A importância de atributos quantifica o quanto cada variável contribuiu para as predições do modelo. Em Random Forest, ela é calculada com base na **redução média de impureza** (Mean Decrease in Impurity) ao longo de todas as árvores. Este resultado conecta o modelo ao objetivo de **Suporte à Decisão**: revela quais fatores a seguradora deve priorizar ao avaliar o risco de um cliente.


In [ ]:
# ─── Feature Importance — Random Forest Otimizado ────────────────────────────
rf_best = grid_rf.best_estimator_

# Nomes das features após OneHotEncoder
ohe_features = list(
    rf_best.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_features)
)
all_features = numeric_features + ohe_features

importances = rf_best.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
colors_imp = ['#e74c3c' if imp == feat_imp.max() else '#3498db' for imp in feat_imp.values]
feat_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title('Figura 8 — Feature Importance (Random Forest Otimizado)', fontweight='bold')
ax.set_xlabel('Importância Relativa')
for i, (val, name) in enumerate(zip(feat_imp.values, feat_imp.index)):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Feature Importance (ranking):")
feat_imp_desc = feat_imp.sort_values(ascending=False)
for i, (feat, imp) in enumerate(feat_imp_desc.items(), 1):
    print(f"  {i:>2}. {feat:<20} {imp:.4f}  {'★' if imp > 0.1 else ''}")


---
## 9. Análise de Resultados

### 9.1 Comparação dos Modelos

A tabela abaixo consolida os resultados de todos os modelos avaliados no conjunto de teste. O critério principal de comparação é o **R²**, complementado pelo **RMSE** e **MAE** para entender a magnitude dos erros.

Os modelos baseados em ensemble (Random Forest e Gradient Boosting) superaram claramente os modelos lineares, evidenciando a presença de **relações não-lineares** e **interações** entre as variáveis preditoras — especialmente a interação entre `smoker`, `age` e `bmi`, observada na EDA.

A otimização de hiperparâmetros via GridSearchCV proporcionou ganhos adicionais, especialmente no Gradient Boosting, confirmando a importância desta etapa no pipeline.

### 9.2 Verificação das Hipóteses

- **H1 (Fumantes pagam mais) → ✅ Confirmada e é o fator mais relevante:** `smoker_yes` aparece como a feature de maior importância no Random Forest, consistente com a diferença de ~3,8× na média de `charges` observada na EDA.

- **H2 (Idade eleva o custo) → ✅ Confirmada:** `age` é a segunda feature mais importante, com correlação de 0.299 com `charges`. O scatter plot revela três faixas de custo distintas por idade — provavelmente segmentadas pelo status de fumante.

- **H3 (IMC eleva o custo) → ✅ Confirmada, com interação:** `bmi` tem importância moderada isoladamente, mas seu efeito é amplificado para fumantes com IMC alto — sugerindo uma **interação não-linear** que os modelos de ensemble capturam melhor que a regressão linear.

- **H4 (Dependentes elevam custo moderadamente) → ✅ Confirmada:** `children` apresenta importância baixa mas positiva, consistente com a baixa correlação (0.068).

- **H5 (Região e sexo têm influência menor) → ✅ Confirmada:** as features derivadas de `region` e `sex` apresentam as menores importâncias no modelo, confirmando que os fatores de risco clínicos dominam a previsão.

### 9.3 Pontos de Atenção

- A distribuição de `charges` apresenta **três clusters** visíveis no scatter de idade, sugerindo que há uma **variável latente não observada** (possivelmente histórico de doenças pré-existentes) que segmenta os segurados. Um modelo ainda mais preciso se beneficiaria dessa informação.
- Os resíduos seguem distribuição aproximadamente normal e sem padrão sistemático, indicando que as premissas do modelo são razoavelmente satisfeitas.
- O dataset é restrito aos EUA e a uma faixa temporal não especificada — a generalização para outros contextos geográficos ou temporais requer validação adicional.

---

## 10. Conclusão e Conexão com Suporte à Decisão

Este trabalho desenvolveu um pipeline completo de Machine Learning para prever o custo individual de seguros médicos. O **Gradient Boosting Otimizado** apresentou o melhor desempenho, com R² superior a 0.87 no conjunto de teste, demonstrando alta capacidade preditiva.

**Contribuição para o Suporte à Decisão na indústria de seguros:**

1. **Precificação de risco personalizada:** o modelo pode ser integrado a um sistema de subscrição para calcular automaticamente o prêmio justo de novos clientes com base no seu perfil.

2. **Priorização de fatores de risco:** a feature importance indica que `smoker`, `age` e `bmi` são os principais determinantes de custo — orientando campanhas de prevenção e critérios de elegibilidade da seguradora.

3. **Alerta de alto custo:** o modelo pode ser usado como componente de um **sistema de alerta precoce**, sinalizando clientes com alto risco de sinistros elevados antes da renovação do contrato.

4. **Equidade e transparência:** a interpretabilidade via feature importance permite que gestores justifiquem as decisões de precificação de forma transparente — aspecto essencial em ambientes regulados.

**Trabalhos futuros** poderiam incluir: análise SHAP para interpretabilidade individual, coleta de variáveis adicionais (histórico de doenças, nível de atividade física) e expansão do dataset para contextos geográficos distintos.

---

## Referências

- Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5–32.
- James, G. et al. (2021). *An Introduction to Statistical Learning* (2ª ed.). Springer.
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR, 12, 2825–2830.
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3ª ed.). O'Reilly.
- Dataset: Medical Cost Personal Dataset. Disponível em: https://github.com/pycaret/pycaret
